# Mini Project 1 — Analysis Notebook

**Your name:**  
**Dataset:**  
**Date:**  

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [24]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *(What is it? Where did it come from? Paste the URL or citation from your MP1a submission.)*

**Why this dataset:** *(One sentence connecting it to your HCD work or research interests.)*

**Three analytical questions:**

1. *(Question 1 from MP1a)*
2. *(Question 2 from MP1a)*
3. *(Question 3 from MP1a)*

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [25]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_csv('your_dataset.csv')  # ← replace with your filename

print(df.shape)
df.head()

(500, 10)


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


In [26]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


In [27]:
# Summary statistics for numeric columns
df.describe()

,id,rating,helpful_votes
count,500.000000,500.000000,500.000000
mean,250.500000,3.946000,23.464000
std,144.481833,1.184013,13.766471
min,1.000000,1.000000,0.000000
25%,125.750000,3.000000,11.000000
50%,250.500000,4.000000,23.500000
75%,375.250000,5.000000,35.000000
max,500.000000,5.000000,47.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *(paste your first research question from MP1a here)*

In [28]:
# Your analysis for Question 1


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [ ]:
# Section 4 — Radial VADER sentiment field with a theme toggle
# - Filter: only posts in record.text with > 50 words.
# - Layout: scatter_polar; radius = |compound|, angle = random spread.
#   Neutral posts sit at the center, opinionated posts fan outward.
# - Color: tertiary discrete (red = negative, grey = neutral, blue = positive).
# - Per-point opacity scales with sentiment strength (min 10%).
# - Dropdown (top-left): "All themes" or any single value from collect_all_themes
#   (posts can belong to multiple themes; the column is ';'-separated).
# - Hover text is wrapped so the post body is actually readable.
#
# pip install vaderSentiment pandas plotly numpy

import html as html_mod
import textwrap
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---- Tunables ---------------------------------------------------------------
MIN_WORDS = 50
MAX_POINTS = 6_000
HOVER_WRAP = 60
HOVER_TEXT_MAX = 1400
NEUTRAL_BAND = 0.05
MIN_OPACITY = 0.10
MAX_OPACITY = 1.00

COLOR_NEGATIVE = "#c0392b"
COLOR_NEUTRAL = "#9aa0a6"
COLOR_POSITIVE = "#2c7fb8"
COLOR_MAP = {
    "negative": COLOR_NEGATIVE,
    "neutral": COLOR_NEUTRAL,
    "positive": COLOR_POSITIVE,
}
CAT_ORDER = ("neutral", "negative", "positive")
THEME_COL = "collect_all_themes"


def hover_post_body(raw):
    """HTML-safe post text with manual line breaks for a readable hover."""
    s = html_mod.escape(str(raw)[:HOVER_TEXT_MAX])
    lines = textwrap.wrap(
        s,
        width=HOVER_WRAP,
        break_long_words=True,
        break_on_hyphens=True,
        replace_whitespace=True,
        drop_whitespace=True,
    )
    body = "<br>".join(lines) if lines else ""
    return body.replace("%", "\uFF05")


# ---- Load -------------------------------------------------------------------
posts_path = None
for fname in ("bsky_posts_collected.csv", "bsky_posts_last_3_years.csv"):
    if Path(fname).exists():
        posts_path = fname
        break
if posts_path is None:
    raise FileNotFoundError(
        "Place bsky_posts_collected.csv or bsky_posts_last_3_years.csv next to this notebook."
    )

df = pd.read_csv(posts_path, low_memory=False)

text_col = "record.text"
time_col = "indexedAt"
for required in (text_col, time_col):
    if required not in df.columns:
        raise ValueError("Missing column " + required + ".")

# ---- Word-count filter ------------------------------------------------------
df["_word_count"] = (
    df[text_col].fillna("").astype(str).str.split().str.len().astype(int)
)
df = df[df["_word_count"] > MIN_WORDS].copy()

# ---- Engagement -------------------------------------------------------------
_eng_pairs = (
    ("record.likeCount", "likeCount"),
    ("record.repostCount", "repostCount"),
    ("record.replyCount", "replyCount"),
    ("record.quoteCount", "quoteCount"),
)
eng_cols = []
for a, b in _eng_pairs:
    if a in df.columns:
        eng_cols.append(a)
    elif b in df.columns:
        eng_cols.append(b)
if eng_cols:
    df["_engagement"] = df[eng_cols].fillna(0).astype(float).sum(axis=1)
else:
    df["_engagement"] = 0.0

# ---- Time (hover-only) ------------------------------------------------------
df["_dt"] = pd.to_datetime(df[time_col], utc=True, errors="coerce")
df["year"] = df["_dt"].dt.year

# ---- Themes (multi-label, ';'-separated) -----------------------------------
if THEME_COL in df.columns:
    df["_themes"] = (
        df[THEME_COL]
        .fillna("")
        .astype(str)
        .apply(lambda s: [t.strip() for t in s.split(";") if t.strip()])
    )
else:
    df["_themes"] = [[] for _ in range(len(df))]

plot_df = df.dropna(subset=[text_col]).copy()

# ---- VADER ------------------------------------------------------------------
analyzer = SentimentIntensityAnalyzer()
plot_df["compound"] = plot_df[text_col].astype(str).map(
    lambda t: analyzer.polarity_scores(t)["compound"]
)

if MAX_POINTS is not None and len(plot_df) > MAX_POINTS:
    plot_df = plot_df.sample(n=MAX_POINTS, random_state=42).reset_index(drop=True)

rng = np.random.default_rng(42)
plot_df["_theta"] = rng.uniform(0.0, 360.0, size=len(plot_df))
plot_df["_r"] = np.abs(plot_df["compound"].astype(float))


def categorize(c):
    if c < -NEUTRAL_BAND:
        return "negative"
    if c > NEUTRAL_BAND:
        return "positive"
    return "neutral"


plot_df["_cat"] = plot_df["compound"].map(categorize)
plot_df["_opacity"] = (
    MIN_OPACITY + (MAX_OPACITY - MIN_OPACITY) * plot_df["_r"].clip(0.0, 1.0)
)
plot_df["_size"] = 6.0 + np.log1p(plot_df["_engagement"].astype(float))

plot_df["_hover_body"] = plot_df[text_col].astype(str).map(hover_post_body)
plot_df["_themes_label"] = plot_df["_themes"].apply(
    lambda lst: ", ".join(lst) if lst else "—"
)
plot_df["_hover"] = (
    "<b>Year</b>: " + plot_df["year"].astype("Int64").astype(str)
    + "<br><b>Sentiment</b>: " + plot_df["_cat"].str.capitalize()
    + "<br><b>VADER compound</b>: " + plot_df["compound"].round(3).astype(str)
    + "<br><b>Words</b>: " + plot_df["_word_count"].astype(int).astype(str)
    + "<br><b>Engagement</b>: " + plot_df["_engagement"].round(0).astype(int).astype(str)
    + "<br><b>Themes</b>: " + plot_df["_themes_label"]
    + "<br><br><b>Post</b><br>" + plot_df["_hover_body"]
)

# ---- Theme list (order by frequency in the visible sample) ------------------
# Filter out non-theme tokens that snuck in via the enrichment pull:
#   - "_enrichment" sentinel (and any other leading-"_" tag)
#   - "reply" / "quote" markers from collect_all_themes="_enrichment;reply;<uri>"
#   - AT-proto URIs (e.g. "at://did:plc:...") used to back-reference viral sources
_NON_THEME_TOKENS = {"reply", "quote"}


def _is_real_theme(token):
    if not token:
        return False
    if token.startswith("_") or token.startswith("at://"):
        return False
    if token in _NON_THEME_TOKENS:
        return False
    return True


theme_counts = Counter()
for tl in plot_df["_themes"]:
    theme_counts.update(t for t in tl if _is_real_theme(t))
THEMES = [t for t, _ in theme_counts.most_common() if _is_real_theme(t)]

# ---- Build traces -----------------------------------------------------------
fig = go.Figure()


def add_sentiment_traces(sub_df, visible):
    """Add three Scatterpolar traces (one per sentiment) for the given subset.
    Returns the indices of the traces just added."""
    indices = []
    for cat in CAT_ORDER:
        s = sub_df[sub_df["_cat"] == cat]
        fig.add_trace(
            go.Scatterpolar(
                r=s["_r"].tolist(),
                theta=s["_theta"].tolist(),
                mode="markers",
                name=cat.capitalize(),
                legendgroup=cat,
                marker=dict(
                    color=COLOR_MAP[cat],
                    size=s["_size"].tolist(),
                    opacity=s["_opacity"].tolist(),
                    line=dict(width=0),
                ),
                customdata=s["_hover"].tolist(),
                hovertemplate="%{customdata}<extra></extra>",
                visible=visible,
                showlegend=True,
            )
        )
        indices.append(len(fig.data) - 1)
    return indices


# "All themes" base traces (visible by default)
all_indices = add_sentiment_traces(plot_df, visible=True)

# One triplet per theme (hidden by default)
theme_trace_indices = {}
theme_counts_in_sample = {}
for theme in THEMES:
    mask = plot_df["_themes"].apply(lambda lst, t=theme: t in lst)
    theme_trace_indices[theme] = add_sentiment_traces(plot_df[mask], visible=False)
    theme_counts_in_sample[theme] = int(mask.sum())

n_total = len(plot_df)
total_traces = len(fig.data)


def make_visibility(visible_indices):
    vis = [False] * total_traces
    for i in visible_indices:
        vis[i] = True
    return vis


def make_title(label, count):
    return (
        "Sentiment as distance from neutral — " + label + "<br>"
        + "<sub>Radius = |VADER compound|; opacity scales with intensity (min 10%); "
        + "n=" + format(count, ",") + " • " + str(posts_path) + "</sub>"
    )


buttons = [
    dict(
        label="All themes (n=" + format(n_total, ",") + ")",
        method="update",
        args=[
            {"visible": make_visibility(all_indices)},
            {"title.text": make_title("All themes", n_total)},
        ],
    )
]
for theme in THEMES:
    buttons.append(
        dict(
            label=theme + " (n=" + format(theme_counts_in_sample[theme], ",") + ")",
            method="update",
            args=[
                {"visible": make_visibility(theme_trace_indices[theme])},
                {"title.text": make_title(theme, theme_counts_in_sample[theme])},
            ],
        )
    )

fig.update_layout(
    title=dict(text=make_title("All themes", n_total), x=0.5, xanchor="center"),
    height=720,
    showlegend=True,
    legend=dict(
        title="Sentiment  (click any item to toggle)",
        orientation="h",
        y=-0.05,
        x=0.5,
        xanchor="center",
        bgcolor="#f6f7f9",
        bordercolor="#bbbbbb",
        borderwidth=1,
        tracegroupgap=22,
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
    hoverlabel=dict(
        align="left",
        font=dict(size=12, family="Helvetica, Arial, sans-serif"),
        namelength=-1,
        bgcolor="white",
        bordercolor="#cccccc",
    ),
    polar=dict(
        bgcolor="white",
        radialaxis=dict(
            title="|compound|  (0 = neutral at center)",
            range=[0, 1],
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            gridcolor="#e5e5e5",
            linecolor="#cccccc",
        ),
        angularaxis=dict(
            showticklabels=False,
            showline=False,
            ticks="",
            gridcolor="#f0f0f0",
        ),
    ),
    paper_bgcolor="white",
    margin=dict(l=40, r=40, t=160, b=140),
    updatemenus=[
        dict(
            type="dropdown",
            direction="up",
            buttons=buttons,
            x=0.0,
            xanchor="left",
            y=-0.12,
            yanchor="top",
            showactive=True,
            bgcolor="white",
            bordercolor="#cccccc",
            font=dict(size=10),
            pad=dict(l=2, r=2, t=2, b=2),
        )
    ],
    annotations=[
        dict(
            text="<b>Theme:</b>",
            x=0.0,
            xref="paper",
            y=-0.10,
            yref="paper",
            xanchor="left",
            yanchor="bottom",
            showarrow=False,
            font=dict(size=12),
        )
    ],
)
fig.show()


### Section 4b — Propagation of one viral post (Sankey)

Where the radial chart shows *posts* as dots, this chart shows the *flow* of
attention out of a single viral post into the kinds of follow-up text it
provoked. The bar thickness encodes engagement, and the downstream flows are
bucketed by composite quality (short vs. substantive, crossed with VADER
sentiment) computed from each descendant's `record.text`.

Two important caveats:

- **Tier 0 to Tier 1** widths come from Bluesky's reported counts on the
  source post (`likeCount`, `repostCount`, `replyCount`, `quoteCount`). These
  are the full, official totals.
- **Tier 1 to Tier 2** widths are *proportional* to the descendant posts we
  happened to capture in this query-collected dataset. The actual mix of
  follow-up quality among all replies and quotes is an estimate, not ground
  truth.
- **Reposts** have no body text on Bluesky and aren't in the dataset as rows,
  so their flow terminates at Tier 1 — we can show the volume of attention,
  but we cannot characterize the quality of a repost.

**If propagation feels sparse:** this CSV was collected by search queries, not by crawling full threads, so only a few replies/quotes from any viral post appear as rows. Ways to get a richer Sankey without changing the CSV:

- **Aggregate many viral roots** — pool the top *K* posts by engagement and sum flows (still sparse per thread, but thicker links overall).
- **Different story, same chart** — flow **total engagement** or **post count** from `collect_all_themes` into sentiment buckets (every row has text; no reply graph needed).
- **Time instead of threads** — bucket posts by month or week of `indexedAt` into themes or sentiment (dense if you use the whole dataset).
- **External data** — use the Bluesky API to fetch full reply chains for one URI (outside this notebook’s CSV).



In [ ]:
# Section 4b — Sankey: propagation of one viral post
# Picks the highest-engagement root post (from the top 20 by engagement) that
# also has descendants captured in this dataset, then routes its attention
# through Reposts / Replies / Quotes, with the Replies + Quotes flows further
# bucketed by composite quality (short vs. substantive x negative/neutral/positive).
#
# pip install vaderSentiment pandas plotly numpy

import html as html_mod
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---- Tunables ---------------------------------------------------------------
TOP_N_FOR_PICKING = 20
SHORT_WORD_THRESHOLD = 30
NEUTRAL_BAND = 0.05
HOVER_WRAP = 60
HOVER_TEXT_MAX = 800
EXAMPLES_PER_BUCKET = 2

# Tier 0 / Tier 1 palette (distinct from sentiment colors so we don't confuse channels)
COLOR_VIRAL = "#2f3640"   # dark slate
COLOR_REPOSTS = "#9aa0a6" # grey
COLOR_REPLIES = "#2c7fb8" # blue
COLOR_QUOTES = "#e67e22"  # orange

# Tier 2 sentiment palette (matches the radial chart)
COLOR_NEGATIVE = "#c0392b"
COLOR_NEUTRAL = "#9aa0a6"
COLOR_POSITIVE = "#2c7fb8"


def hover_post_body(raw, max_chars=HOVER_TEXT_MAX, wrap=HOVER_WRAP):
    """HTML-safe post text with manual line breaks for a readable hover."""
    s = html_mod.escape(str(raw)[:max_chars])
    lines = textwrap.wrap(
        s, width=wrap,
        break_long_words=True, break_on_hyphens=True,
        replace_whitespace=True, drop_whitespace=True,
    )
    body = "<br>".join(lines) if lines else ""
    return body.replace("%", "\uFF05")


def rgba(hex_color, alpha):
    """Convert '#rrggbb' + alpha float -> 'rgba(r,g,b,a)' for Sankey links."""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return "rgba({},{},{},{:.2f})".format(r, g, b, alpha)


# ---- Load (or reuse if upstream cell already loaded it) ---------------------
posts_path = None
for fname in ("bsky_posts_collected.csv", "bsky_posts_last_3_years.csv"):
    if Path(fname).exists():
        posts_path = fname
        break
if posts_path is None:
    raise FileNotFoundError(
        "Place bsky_posts_collected.csv or bsky_posts_last_3_years.csv next to this notebook."
    )

# Always reload here so the cell is standalone (cheap relative to plotting).
full_df = pd.read_csv(posts_path, low_memory=False)

text_col = "record.text"
parent_col = "record.reply.parent.uri"
root_col = "record.reply.root.uri"
quote_col = "record.embed.record.uri"

for c in ("likeCount", "repostCount", "replyCount", "quoteCount"):
    if c not in full_df.columns:
        full_df[c] = 0

full_df["_engagement"] = (
    full_df[["likeCount", "repostCount", "replyCount", "quoteCount"]]
    .fillna(0).astype(float).sum(axis=1)
)
full_df["_word_count"] = (
    full_df[text_col].fillna("").astype(str).str.split().str.len().astype(int)
)

# ---- Pick the viral source post --------------------------------------------
is_root = (
    full_df[parent_col].isna() if parent_col in full_df.columns
    else pd.Series([True] * len(full_df))
)
root_df = full_df[is_root].copy()

top_candidates = root_df.nlargest(TOP_N_FOR_PICKING, "_engagement").copy()


def count_captured_descendants(uri):
    if not isinstance(uri, str) or not uri:
        return 0
    n_replies = int((full_df[root_col] == uri).sum()) if root_col in full_df.columns else 0
    n_quotes = int((full_df[quote_col] == uri).sum()) if quote_col in full_df.columns else 0
    return n_replies + n_quotes


top_candidates["_captured"] = top_candidates["uri"].apply(count_captured_descendants)
# Among the top-engaged roots, pick the one with the most captured descendants.
top_candidates = top_candidates.sort_values(
    ["_captured", "_engagement"], ascending=[False, False]
)
viral = top_candidates.iloc[0]
viral_uri = viral["uri"]

n_reposts = int(viral.get("repostCount", 0) or 0)
n_replies_official = int(viral.get("replyCount", 0) or 0)
n_quotes_official = int(viral.get("quoteCount", 0) or 0)
n_likes = int(viral.get("likeCount", 0) or 0)

# ---- Captured descendants ---------------------------------------------------
replies_df = (
    full_df[full_df[root_col] == viral_uri].copy()
    if root_col in full_df.columns
    else full_df.iloc[0:0].copy()
)
quotes_df = (
    full_df[full_df[quote_col] == viral_uri].copy()
    if quote_col in full_df.columns
    else full_df.iloc[0:0].copy()
)

# Drop the source itself if it accidentally matches itself.
replies_df = replies_df[replies_df["uri"] != viral_uri]
quotes_df = quotes_df[quotes_df["uri"] != viral_uri]

# Score sentiment + word count on each descendant.
analyzer = SentimentIntensityAnalyzer()


def score(df_in):
    df_out = df_in.copy()
    # Coerce NaN/None to "" before VADER. record.text can be missing on
    # notFound/blocked thread stubs and media-only quote posts from the
    # enrichment pull, and StringDtype's .astype(str) leaves NA as NA.
    df_out["_compound"] = (
        df_out[text_col].fillna("").astype(str).map(
            lambda t: analyzer.polarity_scores(t)["compound"] if t else 0.0
        )
    )
    return df_out


replies_df = score(replies_df)
quotes_df = score(quotes_df)

# Drop descendants with no usable text — they'd otherwise pile up as
# "short + neutral" since VADER scores empty strings as 0.
_has_text = lambda d: d[d[text_col].fillna("").astype(str).str.strip().ne("")]
replies_df = _has_text(replies_df)
quotes_df = _has_text(quotes_df)


def bucketize(row):
    substance = "substantive" if row["_word_count"] > SHORT_WORD_THRESHOLD else "short"
    c = row["_compound"]
    if c < -NEUTRAL_BAND:
        sentiment = "negative"
    elif c > NEUTRAL_BAND:
        sentiment = "positive"
    else:
        sentiment = "neutral"
    return substance + " + " + sentiment


for df_in in (replies_df, quotes_df):
    if len(df_in):
        df_in["_bucket"] = df_in.apply(bucketize, axis=1)
    else:
        df_in["_bucket"] = pd.Series([], dtype=str)

# ---- Build Sankey nodes/links ----------------------------------------------
BUCKETS = [
    ("short + negative", COLOR_NEGATIVE),
    ("short + neutral", COLOR_NEUTRAL),
    ("short + positive", COLOR_POSITIVE),
    ("substantive + negative", COLOR_NEGATIVE),
    ("substantive + neutral", COLOR_NEUTRAL),
    ("substantive + positive", COLOR_POSITIVE),
]

VIRAL_NODE = 0
REPOSTS_NODE = 1
REPLIES_NODE = 2
QUOTES_NODE = 3
BUCKET_NODE_START = 4
BUCKET_INDEX = {label: BUCKET_NODE_START + i for i, (label, _) in enumerate(BUCKETS)}

viral_text = str(viral.get(text_col, ""))
viral_first_line = viral_text.split("\n", 1)[0]
if len(viral_first_line) > 80:
    viral_first_line = viral_first_line[:77] + "..."
viral_handle = str(viral.get("author.handle", "unknown"))

node_labels = [
    "@" + viral_handle + ": \"" + viral_first_line + "\"",
    "Reposts ({:,})".format(n_reposts),
    "Replies ({:,})".format(n_replies_official),
    "Quotes ({:,})".format(n_quotes_official),
] + [label for label, _ in BUCKETS]

node_colors = [
    COLOR_VIRAL, COLOR_REPOSTS, COLOR_REPLIES, COLOR_QUOTES,
] + [color for _, color in BUCKETS]

# Node hovers
viral_hover = (
    "<b>Viral source</b><br>"
    "<b>Author</b>: @" + viral_handle + "<br>"
    "<b>Total engagement</b>: " + format(int(viral["_engagement"]), ",") + "<br>"
    "<b>Likes</b>: " + format(n_likes, ",") + "<br>"
    "<b>Reposts</b>: " + format(n_reposts, ",") + "<br>"
    "<b>Replies</b>: " + format(n_replies_official, ",") + "<br>"
    "<b>Quotes</b>: " + format(n_quotes_official, ",") + "<br>"
    "<b>Captured descendants in dataset</b>: "
    + format(len(replies_df) + len(quotes_df), ",") + "<br><br>"
    "<b>Post</b><br>" + hover_post_body(viral_text, max_chars=600)
)


def capture_hover(label, official_count, captured_count):
    rate = (captured_count / official_count * 100.0) if official_count else 0.0
    return (
        "<b>" + label + "</b><br>"
        "<b>Bluesky-reported</b>: " + format(official_count, ",") + "<br>"
        "<b>Captured in this dataset</b>: " + format(captured_count, ",") + "<br>"
        "<b>Capture rate</b>: " + ("{:.1f}%".format(rate))
    )


reposts_hover = (
    "<b>Reposts</b><br>"
    "<b>Bluesky-reported</b>: " + format(n_reposts, ",") + "<br>"
    "Bluesky reposts carry no body text, so quality cannot be measured."
)
replies_hover = capture_hover("Replies", n_replies_official, len(replies_df))
quotes_hover = capture_hover("Quotes", n_quotes_official, len(quotes_df))


def bucket_hover(label, replies_in_bucket, quotes_in_bucket):
    combined = pd.concat([replies_in_bucket, quotes_in_bucket], ignore_index=True)
    if combined.empty:
        return "<b>" + label + "</b><br>No captured descendants in this bucket."
    mean_words = combined["_word_count"].mean()
    mean_compound = combined["_compound"].mean()
    sample_texts = combined[text_col].astype(str).head(EXAMPLES_PER_BUCKET).tolist()
    snippets = []
    for t in sample_texts:
        snippets.append("- " + hover_post_body(t, max_chars=240, wrap=54))
    hover = (
        "<b>" + label + "</b><br>"
        "<b>Captured descendants</b>: " + format(len(combined), ",") + "<br>"
        "<b>Mean word count</b>: " + "{:.0f}".format(mean_words) + "<br>"
        "<b>Mean VADER compound</b>: " + "{:+.3f}".format(mean_compound) + "<br><br>"
        "<b>Examples</b><br>" + "<br><br>".join(snippets)
    )
    return hover


bucket_hovers = []
for label, _ in BUCKETS:
    r_in = replies_df[replies_df["_bucket"] == label] if len(replies_df) else replies_df
    q_in = quotes_df[quotes_df["_bucket"] == label] if len(quotes_df) else quotes_df
    bucket_hovers.append(bucket_hover(label, r_in, q_in))

node_hovers = [
    viral_hover, reposts_hover, replies_hover, quotes_hover,
] + bucket_hovers

# ---- Links ------------------------------------------------------------------
link_sources = []
link_targets = []
link_values = []
link_colors = []
link_hovers = []

# Tier 0 -> Tier 1 (widths = official Bluesky counts)
if n_reposts > 0:
    link_sources.append(VIRAL_NODE)
    link_targets.append(REPOSTS_NODE)
    link_values.append(n_reposts)
    link_colors.append(rgba(COLOR_REPOSTS, 0.45))
    link_hovers.append(
        "<b>Viral -> Reposts</b><br>" + format(n_reposts, ",") + " reposts (no text)"
    )
if n_replies_official > 0:
    link_sources.append(VIRAL_NODE)
    link_targets.append(REPLIES_NODE)
    link_values.append(n_replies_official)
    link_colors.append(rgba(COLOR_REPLIES, 0.45))
    link_hovers.append(
        "<b>Viral -> Replies</b><br>" + format(n_replies_official, ",") + " replies on Bluesky<br>"
        + format(len(replies_df), ",") + " captured in this dataset"
    )
if n_quotes_official > 0:
    link_sources.append(VIRAL_NODE)
    link_targets.append(QUOTES_NODE)
    link_values.append(n_quotes_official)
    link_colors.append(rgba(COLOR_QUOTES, 0.45))
    link_hovers.append(
        "<b>Viral -> Quotes</b><br>" + format(n_quotes_official, ",") + " quotes on Bluesky<br>"
        + format(len(quotes_df), ",") + " captured in this dataset"
    )


def add_tier2_flows(channel_df, source_node, channel_label, official_count, channel_color):
    """Allocate channel's official width across Tier 2 buckets proportionally."""
    if official_count <= 0 or len(channel_df) == 0:
        return
    bucket_counts = channel_df["_bucket"].value_counts()
    captured_total = bucket_counts.sum()
    if captured_total == 0:
        return
    for label, _ in BUCKETS:
        captured_in_bucket = int(bucket_counts.get(label, 0))
        if captured_in_bucket == 0:
            continue
        width = official_count * (captured_in_bucket / captured_total)
        if width <= 0:
            continue
        link_sources.append(source_node)
        link_targets.append(BUCKET_INDEX[label])
        link_values.append(width)
        link_colors.append(rgba(channel_color, 0.30))
        link_hovers.append(
            "<b>" + channel_label + " -> " + label + "</b><br>"
            "<b>Allocated width</b>: " + "{:,.0f}".format(width) + "<br>"
            "<b>Captured in this dataset</b>: " + str(captured_in_bucket)
            + " of " + str(captured_total)
            + " (" + "{:.1f}%".format(100 * captured_in_bucket / captured_total) + ")"
        )


add_tier2_flows(replies_df, REPLIES_NODE, "Replies", n_replies_official, COLOR_REPLIES)
add_tier2_flows(quotes_df, QUOTES_NODE, "Quotes", n_quotes_official, COLOR_QUOTES)

# ---- Figure -----------------------------------------------------------------
# Pin nodes to three columns so the flow reads: one source -> Reposts / Replies / Quotes -> quality.
# Without explicit x/y, Plotly's auto layout often stacks the viral node beside its children.
_col_left, _col_mid, _col_right = 0.02, 0.30, 0.58
_y_tier1 = [0.22, 0.50, 0.78]  # nodes 1-3: Reposts, Replies, Quotes
_y_tier2 = np.linspace(0.10, 0.90, len(BUCKETS)).tolist()
node_x = [_col_left] + [_col_mid] * 3 + [_col_right] * len(BUCKETS)
node_y = [0.5] + _y_tier1 + _y_tier2

fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="fixed",
            node=dict(
                pad=22,
                thickness=18,
                line=dict(color="#cccccc", width=0.5),
                label=node_labels,
                color=node_colors,
                x=node_x,
                y=node_y,
                customdata=node_hovers,
                hovertemplate="%{customdata}<extra></extra>",
            ),
            link=dict(
                source=link_sources,
                target=link_targets,
                value=link_values,
                color=link_colors,
                customdata=link_hovers,
                hovertemplate="%{customdata}<extra></extra>",
            ),
        )
    ]
)

title_text = (
    "Propagation of one viral AI post on Bluesky<br>"
    "<sub>Source: @" + viral_handle + " - total engagement "
    + format(int(viral["_engagement"]), ",") + " - "
    + format(len(replies_df) + len(quotes_df), ",")
    + " descendants captured in this dataset</sub>"
)

fig.update_layout(
    title=dict(text=title_text, x=0.5, xanchor="center"),
    width=1200,
    height=640,
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=110, b=20),
    hoverlabel=dict(
        align="left",
        font=dict(size=12, family="Helvetica, Arial, sans-serif"),
        namelength=-1,
        bgcolor="white",
        bordercolor="#cccccc",
    ),
    font=dict(size=12),
)
fig.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.